In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ast
import re

matrix = pd.read_csv("prs_conditionalOR_matrix_260217_childrencode.csv")
matrix.columns = matrix.columns.str.replace("_hmPOS_GRCh38", "", regex=False)
icd_info = pd.read_csv("../../ICD10CM_childrencode_WGS.csv")
pgs_map = pd.read_csv("../../disease_preprocess/pgs_id_list_260217.csv") 
loinc_pgs_map = pd.read_csv("../../measurement_preprocess/pgs_id_list_260225.csv") 

### Basic Info

In [2]:
# basic information of raw result

# select information
target_icd_list = []
target_pgs_list = []
for idx, row in pgs_map.iterrows():
    pgs_ids = ast.literal_eval(row['pgs_ids'])
    target_icd_list.append(row['icd'])
    target_pgs_list.extend(pgs_ids)

# calculate information
calc_icd_list = matrix["trait"]
calc_pgs_list = matrix.columns.drop("trait").tolist()

print("Number of target Traits (ICD10CM children code):", len(set(target_icd_list)))
print("Number of target PGS (PGS ids):", len(set(target_pgs_list)))
print("Number of calculated Traits (ICD10CM children code):", len(calc_icd_list))
print("Number of calculated PGS (PGS ids):", len(calc_pgs_list))

Number of target Traits (ICD10CM children code): 173
Number of target PGS (PGS ids): 1839
Number of calculated Traits (ICD10CM children code): 170
Number of calculated PGS (PGS ids): 3087


### QC

In [3]:
# filter case number bigger than 200
print("Before filter:", matrix.shape, len(matrix["trait"]))

trait_case_count = pd.read_csv("trait_case_count_260217_childrencode.csv").drop_duplicates("ICD10_code")

# merge case count
matrix_merged = matrix.merge(trait_case_count, left_on="trait", right_on="ICD10_code", how="left")

# list of include and exclude trait
included_traits = matrix_merged.loc[
    matrix_merged["case_count"] >= 200, "trait"
].tolist()

excluded_traits = matrix_merged.loc[
    matrix_merged["case_count"] < 200, "trait"
].tolist()

print("Included trait number:", len(included_traits))
print("Excluded trait number:", len(excluded_traits))
print("Excluded trait:", excluded_traits)

# filter
matrix = matrix_merged[matrix_merged["case_count"] >= 200]
matrix = matrix.drop(columns=["ICD10_code", "case_count"])

print("After filter:", matrix.shape, len(matrix["trait"]))

Before filter: (170, 3088) 170
Included trait number: 148
Excluded trait number: 22
Excluded trait: ['C119', 'C323', 'C33', 'C6290', 'C9100', 'D282', 'D863', 'E035', 'E05', 'H8093', 'I069', 'I2542', 'I256', 'J84115', 'M0230', 'M0800', 'M0840', 'M368', 'M729', 'N028', 'O1490', 'O360990']
After filter: (148, 3088) 148


In [4]:
# generate a mapping between pgs id and icd id, change colname, set trait information as index
from collections import defaultdict
print("Before filter:", matrix.shape)

# get mapping between PGS id and icd code
pgs2icds = defaultdict(set)
for idx, row in pgs_map.iterrows():
    pgs_ids = ast.literal_eval(row['pgs_ids'])
    icd_code = row['icd']

    for pgs in pgs_ids:
        pgs2icds[pgs].add(icd_code)

for idx, row in loinc_pgs_map.iterrows():
    pgs_ids = ast.literal_eval(row['pgs_ids'])
    icd_code = row['loinc']

    for pgs in pgs_ids:
        pgs2icds[pgs].add(icd_code)

# rename each col
matrix = matrix.set_index("trait")
new_columns = []
for col in matrix.columns:
    if col in pgs2icds:
        icd_list = "+".join(pgs2icds[col])
        new_columns.append(f"{icd_list}__{col}")
    else:
        new_columns.append(f"NA__{col}")

matrix.columns = new_columns
print("After filter:", matrix.shape)

Before filter: (148, 3088)
After filter: (148, 3087)


In [5]:
# check how many pgs for each trait is calculated
matrix_df = matrix.copy()
prs_count = {}

for trait in matrix_df.index:
    count = 0
    
    for col in matrix_df.columns:
        if "__" not in col:
            continue
        
        icd_part = col.split("__")[0]      # C44+C43
        icd_list = icd_part.split("+")     # ["C44","C43"]

        if trait in icd_list:
            count += 1

    prs_count[trait] = count

prs_count_df = pd.DataFrame.from_dict(
    prs_count,
    orient="index",
    columns=["calc_prs_count"]
)

prs_count_df = prs_count_df.reset_index()
prs_count_df = prs_count_df.rename(columns={"index": "icd"})

In [6]:
# save data after QC
result = pgs_map.merge(prs_count_df, on="icd")
result["include_in_analysis"] = result["icd"].isin(included_traits).astype("int8")
result = result.merge(
    trait_case_count[["ICD10_code", "case_count"]],
    left_on="icd",
    right_on="ICD10_code",
    how="left"
).rename(columns={"case_count": "case_num"})
result = result.drop(columns=["pgs_urls", "ICD10_code"])
result.to_csv("prs_conditionalOR_metadata_260217_childrencode.csv", index=False)
matrix.to_csv("prs_conditionalOR_matrix_260217_childrencode_qc.csv", index=True)
print("Data shape after QC:", matrix.shape)

Data shape after QC: (148, 3087)


### Save Rank Matrix

In [7]:
import numpy as np
matrix = pd.read_csv("prs_conditionalOR_matrix_260217_childrencode_qc.csv")
matrix = matrix.set_index("trait")

# check how many pgs for each trait is calculated
matrix_df = matrix.copy()
prs_count = {}

for trait in matrix_df.index:
    count = 0
    
    for col in matrix_df.columns:
        if "__" not in col:
            continue
        
        icd_part = col.split("__")[0]      # C44+C43
        icd_list = icd_part.split("+")     # ["C44","C43"]

        if trait in icd_list:
            count += 1

    if trait == "C50":
        print(count)
    prs_count[trait] = count

prs_count_df = pd.DataFrame.from_dict(
    prs_count,
    orient="index",
    columns=["calc_prs_count"]
)

prs_count_df = prs_count_df.reset_index()
prs_count_df = prs_count_df.rename(columns={"index": "icd"})

# save ranking matrix 
rank_rows = []
for trait in matrix_df.index: 
    row = matrix_df.loc[trait].sort_values(ascending=False) 
    rank_rows.append(row.index.tolist())
rank_df = pd.DataFrame(
    rank_rows,
    index=matrix_df.index,
    columns=[f"rank_{i+1}" for i in range(matrix_df.shape[1])]
)
rank_df.to_csv("prs_conditionalOR_matrix_260217_childrencode_rankmatrix.csv", index=True)